# RU OCR: speed benchmark — Paddle (GPU, CPU), ONNX Runtime, OpenVINO

The `CKPT` checkpoint of `CKPT_DATASET` is exported (Paddle inference model, and ONNX from a CPU export) and read
on the lines of `LABELS` by each backend through PaddleX `create_model`, as a user would run it:

| Backend | Model | Device |
|---|---|---|
| `paddle gpu` | Paddle inference | one GPU (only in a GPU session) |
| `paddle cpu` | Paddle inference | CPU |
| `onnxruntime cpu` | ONNX | CPU, `CPUExecutionProvider` |
| `openvino cpu` | ONNX | CPU, ONNX Runtime `OpenVINOExecutionProvider` |

- With `SORT_BY_WIDTH` every batch size above 1 runs twice: lines in file order, and sorted by width/height so
  that a batch holds lines of similar width (each batch is padded to its widest line).
- Each backend × batch size runs in its own process. The images are decoded into memory first, so disk and
  decoding are not timed; the first `WARMUP` calls are not timed either.
- Measured: lines/s, ms per line, ms per `predict` call (one batch: mean, p50, p95), model load time, and accuracy on
  the same lines (exact match, spaces count) as a check that every backend reads the same.
- Thread counts are the libraries' defaults; the hardware header says how many cores the process may use.

**Output** (`/kaggle/working`): `benchmark.md` (hardware header + table), `benchmark.tsv`, `hardware.json`.

**Input**: the checkpoint dataset (`ru-ocr-checkpoints-ft4`) and `ru-ocr-val5k-tar` (the 5 000 crops of `all_val5k`
in one `.tarpack`, 26 MB, with `all_val5k_w_h.txt` and `all_val5k_clean_w_h.txt`). Smoke run: attach `ru-ocr-smoke` (100 lines of `all_val5k_clean_w_h.txt` with their crops) instead of the full
crops and labels. **Settings**: Accelerator GPU T4 x2 (all four backends) or None (CPU backends only), Internet on.
No secret needed. Save & Run All.

In [ ]:
REPO = "https://github.com/ErshovVE/PaddleOCR.git"  # cloned for the config and tools/export_model.py
BRANCH = "feat/ru-ppocrv6-small-training"
CKPT = "best_accuracy"   # checkpoint prefix in CKPT_DATASET
CKPT_DATASET = "ru-ocr-checkpoints-ft4"
CONFIG = "ru_ocr/configs/ru_PP-OCRv6_small_rec.yml"
MODEL_NAME = "PP-OCRv6_small_rec"
LABELS = "all_val5k_clean_w_h.txt"  # found anywhere under /kaggle/input (ru-ocr-labels or ru-ocr-smoke)
LIMIT = None             # only the first LIMIT lines (None = all)
BATCHES = [1, 4, 8]      # batch sizes, each a separate run
SORT_BY_WIDTH = True     # batches above 1 also run on lines sorted by width/height (similar widths per batch)
WARMUP = 3               # predict calls before timing starts
BACKENDS = ["paddle gpu", "paddle cpu", "onnxruntime cpu", "openvino cpu"]  # "paddle gpu" is skipped without a GPU
MODEL_DIR = "/kaggle/working/model"
import os, re, shutil, subprocess, sys, time
HAS_GPU = shutil.which("nvidia-smi") is not None and subprocess.run(["nvidia-smi", "-L"], capture_output=True).returncode == 0
print("GPU session" if HAS_GPU else "CPU session: paddle gpu is skipped")

## Hardware

In [ ]:
def out(cmd):
    res = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    return res.stdout.strip() if res.returncode == 0 else ""


def cgroup_cpus():
    """CPU quota of the container in cores (cgroup v2 cpu.max or v1 cfs quota), None when unlimited."""
    try:
        quota, period = open("/sys/fs/cgroup/cpu.max").read().split()
        return None if quota == "max" else int(quota) / int(period)
    except (OSError, ValueError):
        pass
    try:
        quota = int(open("/sys/fs/cgroup/cpu/cpu.cfs_quota_us").read())
        period = int(open("/sys/fs/cgroup/cpu/cpu.cfs_period_us").read())
        return None if quota <= 0 else quota / period
    except (OSError, ValueError):
        return None


lscpu = dict((k.strip(), v.strip()) for k, v in (line.split(":", 1) for line in out("lscpu").splitlines() if ":" in line))
meminfo = dict(line.split(":", 1) for line in open("/proc/meminfo").read().splitlines())
affinity = len(os.sched_getaffinity(0))
quota = cgroup_cpus()
HW = {
    "gpu": out("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader").splitlines() if HAS_GPU else [],
    "gpu_driver": out("nvidia-smi --query-gpu=driver_version --format=csv,noheader").split("\n")[0] if HAS_GPU else "",
    "cuda_driver": (re.findall(r"CUDA Version: ([\d.]+)", out("nvidia-smi")) or [""])[0] if HAS_GPU else "",
    "cpu_model": lscpu.get("Model name", ""),
    "cpu_sockets": lscpu.get("Socket(s)", ""),
    "cpu_cores_per_socket": lscpu.get("Core(s) per socket", ""),
    "cpu_threads_per_core": lscpu.get("Thread(s) per core", ""),
    "cpu_logical_host": os.cpu_count(),
    "cpu_affinity": affinity,
    "cpu_quota": quota,
    "cpu_available": min(affinity, quota) if quota else affinity,  # what this process may actually use
    "ram_total_gb": round(int(meminfo["MemTotal"].split()[0]) / 2**20, 1),
    "ram_available_gb": round(int(meminfo["MemAvailable"].split()[0]) / 2**20, 1),
    "python": sys.version.split()[0],
}
for k, v in HW.items():
    print(f"{k:22} {v}")

## Paddle, PaddleX, ONNX Runtime with OpenVINO, code

In [ ]:
import shlex, signal, threading


def sh(cmd, timeout=1200, attempts=3):
    """Run a shell command with its output in the cell. Some installs stall forever with an idle CPU: a run is
    killed with its whole process group after `timeout` seconds and retried."""
    for attempt in range(1, attempts + 1):
        print(time.strftime("%H:%M:%S"), f"[{attempt}/{attempts}]", cmd, flush=True)
        proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                                start_new_session=True)
        timer = threading.Timer(timeout, os.killpg, (proc.pid, signal.SIGKILL))
        timer.start()
        for line in proc.stdout:
            print(line, end="", flush=True)
        hung = not timer.is_alive()
        timer.cancel()
        if proc.wait() == 0:
            return
        print(time.strftime("%H:%M:%S"), f"hung for {timeout} s, killed" if hung else f"exit code {proc.returncode}", flush=True)
    raise RuntimeError(f"failed {attempts} times: {cmd}")


if HAS_GPU:
    paddle_pkg = "paddlepaddle-gpu==3.3.1 -i https://www.paddlepaddle.org.cn/packages/stable/cu126/"
else:
    paddle_pkg = "paddlepaddle==3.3.1"
PIP = f"{shlex.quote(sys.executable)} -m pip install --progress-bar off --timeout 120 --retries 10"
from importlib.metadata import PackageNotFoundError, version
try:
    IMAGE_NCCL = version("nvidia-nccl-cu12")
except PackageNotFoundError:
    IMAGE_NCCL = None
sh(f"{PIP} {paddle_pkg}", attempts=5)
# paddlepaddle-gpu (cu126) pins an older nvidia-nccl-cu12 over the image's one, and the image's torch then fails on
# import (paddlex imports torch through modelscope): the image's version goes back
if HAS_GPU and IMAGE_NCCL and version("nvidia-nccl-cu12") != IMAGE_NCCL:
    sh(f"{PIP} --no-deps --force-reinstall nvidia-nccl-cu12=={IMAGE_NCCL}")
sh(f"{PIP} 'paddlex[ocr-core]>=3.7,<3.8' onnxruntime-openvino")
# ONNX: paddle2onnx has wheels only for Python <= 3.12 and does not take an export made with paddlepaddle-gpu, so
# a CPU paddle in a Python 3.12 venv exports the checkpoint once more for the conversion
sh(f"{PIP} uv")
sh("uv venv --clear --seed --python 3.12 /tmp/py312")
sh("uv pip install --python /tmp/py312/bin/python paddlepaddle==3.3.1 'paddlex[ocr-core]>=3.7,<3.8'")
sh("/tmp/py312/bin/paddlex --install paddle2onnx")
import onnxruntime
print("ONNX Runtime providers:", onnxruntime.get_available_providers())
res = subprocess.run([sys.executable, "-c", "import torch, paddle; print('torch', torch.__version__, 'paddle', "
                      "paddle.__version__, 'cuda', paddle.device.is_compiled_with_cuda())"], capture_output=True, text=True)
print(res.stdout.strip(), res.stderr.strip()[-2000:] if res.returncode else "")
assert res.returncode == 0, "torch or paddle does not import, see above"

In [ ]:
%cd /kaggle/working
!rm -rf PaddleOCR && git clone -q --depth 1 --branch {BRANCH} {REPO}
%cd /kaggle/working/PaddleOCR
!git log --oneline -1
sh(f"{PIP} -r requirements.txt")
sh("uv pip install --python /tmp/py312/bin/python -r requirements.txt")  # tools/export_model.py in the venv

## Data, labels, weights
Crop archives `*.tarpack` are unpacked into `/tmp/ru_root`; folders of single crops (`<...>/stroyinf/<folder>` with
`crops/`) are linked there.

In [ ]:
ROOT = "/tmp/ru_root"
os.makedirs(os.path.join(ROOT, "stroyinf"), exist_ok=True)
for top, dirs, files in os.walk("/kaggle/input"):
    for name in sorted(f for f in files if f.endswith(".tarpack")):
        start = time.time()
        subprocess.run(["tar", "-xf", os.path.join(top, name), "-C", ROOT], check=True)
        print(f"unpacked {name} in {time.time() - start:.0f} s", flush=True)
labels_path, weights = None, []
for top, dirs, files in os.walk("/kaggle/input"):
    if "crops" in dirs:
        dirs.remove("crops")
        if os.path.basename(os.path.dirname(top)) == "stroyinf":
            link = os.path.join(ROOT, "stroyinf", os.path.basename(top))
            if not os.path.exists(link):
                os.symlink(top, link)
    if LABELS in files:
        labels_path = os.path.join(top, LABELS)
    if CKPT + ".pdparams" in files and CKPT_DATASET in top.split(os.sep):
        weights.append(os.path.join(top, CKPT))
assert labels_path and len(weights) == 1, f"attach a dataset with {LABELS} and {CKPT_DATASET}: {labels_path}, {weights}"
WEIGHTS = weights[0]
with open(labels_path, encoding="utf-8") as f:
    n_lines = sum(1 for line in f if line.strip())
print("labels:", labels_path, n_lines, "lines", f"(first {LIMIT} used)" if LIMIT else "")
print("weights:", WEIGHTS)

## Export: checkpoint -> Paddle inference model, ONNX
`model/inference` is exported by the session's paddle (GPU build in a GPU session) and serves both Paddle backends;
`model/inference_cpu` is exported by the CPU paddle of the venv only to be converted into `model/onnx`.

In [ ]:
export = ["tools/export_model.py", "-c", CONFIG, "-o", f"Global.pretrained_model={WEIGHTS}"]
subprocess.run([sys.executable, *export, f"Global.save_inference_dir={MODEL_DIR}/inference"], check=True,
               stdout=subprocess.DEVNULL)
subprocess.run(["/tmp/py312/bin/python", *export, "Global.use_gpu=false",
                f"Global.save_inference_dir={MODEL_DIR}/inference_cpu"], check=True, stdout=subprocess.DEVNULL)
# PATH: paddlex looks for the paddle2onnx executable of the venv
env = dict(os.environ, PATH="/tmp/py312/bin:" + os.environ["PATH"])
subprocess.run(["paddlex", "--paddle2onnx", "--paddle_model_dir", f"{MODEL_DIR}/inference_cpu",
                "--onnx_model_dir", f"{MODEL_DIR}/onnx"], env=env, check=True)
for d in ("inference", "onnx"):
    print(d, sorted(os.listdir(f"{MODEL_DIR}/{d}")))

## Benchmark
`bench_worker.py` runs one backend with one batch size in its own process and writes a JSON result.

In [ ]:
%%writefile /kaggle/working/bench_worker.py
# one run: python bench_worker.py <labels> <root> <model kwargs json> <model name> <batch> <warmup> <limit> <sort> <out json>
# sort = 1: lines go in order of width/height (similar widths per batch); texts are returned in file order
import json
import os
import statistics
import sys
import time

import cv2
from paddlex import create_model

labels, root, kwargs, model_name, batch, warmup, limit, sort, out = sys.argv[1:10]
batch, warmup, limit, sort = int(batch), int(warmup), int(limit), sort == "1"
with open(labels, encoding="utf-8") as f:
    rows = [line.rstrip("\n").split("\t")[:2] for line in f if line.strip()]
if limit:
    rows = rows[:limit]
images = [cv2.imread(os.path.join(root, path)) for path, _ in rows]  # decoded up front: disk and decoding not timed
missing = [path for (path, _), image in zip(rows, images) if image is None]
assert not missing, f"{len(missing)} crops not found, e.g. {missing[:3]}"

start = time.perf_counter()
model = create_model(model_name=model_name, **json.loads(kwargs))
load_s = time.perf_counter() - start
order = list(range(len(images)))
if sort:
    order.sort(key=lambda i: images[i].shape[1] / images[i].shape[0])
chunks = [[images[i] for i in order[k : k + batch]] for k in range(0, len(order), batch)]
for chunk in chunks[:warmup]:
    list(model.predict(chunk, batch_size=batch))

calls, texts = [], []
start = time.perf_counter()
for chunk in chunks:
    t = time.perf_counter()
    res = list(model.predict(chunk, batch_size=batch))
    calls.append(time.perf_counter() - t)
    texts += [r["rec_text"] for r in res]
total = time.perf_counter() - start
texts = [text for _, text in sorted(zip(order, texts))]  # back to file order

ms = sorted(c * 1000 for c in calls)
labels_ = [label for _, label in rows]
result = {
    "lines": len(rows),
    "load_s": round(load_s, 2),
    "total_s": round(total, 2),
    "lines_per_s": round(len(rows) / total, 1),
    "ms_per_line": round(1000 * total / len(rows), 2),
    "ms_per_call_mean": round(statistics.mean(ms), 2),
    "ms_per_call_p50": round(ms[len(ms) // 2], 2),
    "ms_per_call_p95": round(ms[min(len(ms) - 1, int(0.95 * len(ms)))], 2),
    "acc": round(sum(t == l for t, l in zip(texts, labels_)) / len(rows), 4),
    "acc_rstrip": round(sum(t.rstrip() == l for t, l in zip(texts, labels_)) / len(rows), 4),  # batch padding can add a trailing space
}
with open(out, "w", encoding="utf-8") as f:
    json.dump({"result": result, "texts": texts}, f, ensure_ascii=False)
print(json.dumps(result))

In [ ]:
import json
import pandas as pd

ov = {"providers": ["OpenVINOExecutionProvider"], "provider_options": {"device_type": "CPU"}}
specs = {  # backend -> (create_model kwargs, environment of the worker)
    "paddle gpu": (dict(model_dir=f"{MODEL_DIR}/inference", device="gpu:0"), {"CUDA_VISIBLE_DEVICES": "0"}),
    "paddle cpu": (dict(model_dir=f"{MODEL_DIR}/inference", device="cpu"), {"CUDA_VISIBLE_DEVICES": ""}),
    "onnxruntime cpu": (dict(model_dir=f"{MODEL_DIR}/onnx", device="cpu", engine="onnxruntime",
                             engine_config={"providers": ["CPUExecutionProvider"]}), {"CUDA_VISIBLE_DEVICES": ""}),
    "openvino cpu": (dict(model_dir=f"{MODEL_DIR}/onnx", device="cpu", engine="onnxruntime", engine_config=ov),
                     {"CUDA_VISIBLE_DEVICES": ""}),
}
runs = [b for b in BACKENDS if HAS_GPU or b != "paddle gpu"]
rows, texts = [], {}
for backend in runs:
    kwargs, env = specs[backend]
    for batch, sort in [(b, s) for b in BATCHES for s in ([False, True] if SORT_BY_WIDTH and b > 1 else [False])]:
        tag = f"{backend.replace(' ', '_')}_b{batch}" + ("_sorted" if sort else "")
        out_json = f"/kaggle/working/bench_{tag}.json"
        res = subprocess.run([sys.executable, "/kaggle/working/bench_worker.py", labels_path, ROOT, json.dumps(kwargs),
                              MODEL_NAME, str(batch), str(WARMUP), str(LIMIT or 0), str(int(sort)), out_json],
                             capture_output=True, text=True, env=dict(os.environ, **env))
        assert res.returncode == 0, f"{backend} batch {batch} failed:\n{(res.stdout + res.stderr)[-3000:]}"
        with open(out_json, encoding="utf-8") as f:
            data = json.load(f)
        texts[(backend, batch, sort)] = data["texts"]
        rows.append({"backend": backend, "batch": batch, "sorted": sort, **data["result"]})
        r = data["result"]
        print(f"{backend:16} batch {batch:3}{' sorted' if sort else '       '}: {r['lines_per_s']:8.1f} lines/s, {r['ms_per_line']:7.2f} ms/line, "
              f"acc {r['acc']:.4f}", flush=True)
bench = pd.DataFrame(rows)
ref = texts[(runs[0], BATCHES[0], False)]
bench["same_as_" + runs[0].replace(" ", "_") + f"_b{BATCHES[0]}"] = [
    round(sum(a == b for a, b in zip(texts[(r.backend, r.batch, r.sorted)], ref)) / len(ref), 4) for r in bench.itertuples()]
bench

In [ ]:
from importlib.metadata import PackageNotFoundError, version


def ver(name):
    try:
        return version(name)
    except PackageNotFoundError:
        return None


HW["versions"] = {name: ver(name) for name in ("paddlepaddle-gpu", "paddlepaddle", "paddlex", "onnxruntime-openvino",
                                                "onnxruntime", "openvino", "opencv-python") if ver(name)}
HW["paddle2onnx (venv)"] = out("/tmp/py312/bin/python -c 'import paddle2onnx; print(paddle2onnx.__version__)'")
HW["labels"] = f"{LABELS}, {int(bench.lines.iloc[0])} lines"
with open("/kaggle/working/hardware.json", "w", encoding="utf-8") as f:
    json.dump(HW, f, ensure_ascii=False, indent=1)
bench.to_csv("/kaggle/working/benchmark.tsv", sep="\t", index=False)

gpus = "; ".join(HW["gpu"]) or "none"
header = [
    "# RU OCR speed benchmark",
    "",
    f"- GPU: {gpus}" + (f" (driver {HW['gpu_driver']}, CUDA {HW['cuda_driver']}); Paddle uses one card" if HW["gpu"] else ""),
    f"- CPU: {HW['cpu_model']}, {HW['cpu_sockets']} socket(s) x {HW['cpu_cores_per_socket']} cores x "
    f"{HW['cpu_threads_per_core']} threads; {HW['cpu_logical_host']} logical on the host, "
    f"**{HW['cpu_available']} available to the process** (affinity {HW['cpu_affinity']}, quota {HW['cpu_quota']})",
    f"- RAM: {HW['ram_total_gb']} GB ({HW['ram_available_gb']} GB available)",
    f"- Python {HW['python']}; " + ", ".join(f"{k} {v}" for k, v in HW["versions"].items())
    + f"; paddle2onnx {HW['paddle2onnx (venv)']}",
    f"- Model: {CKPT_DATASET}/{CKPT} ({MODEL_NAME}); lines: {HW['labels']}; images decoded in memory; "
    f"{WARMUP} warm-up calls not timed; library default threads",
    "",
]
with open("/kaggle/working/benchmark.md", "w", encoding="utf-8") as f:
    table = ["| " + " | ".join(bench.columns) + " |", "|" + "---|" * len(bench.columns)]
    table += ["| " + " | ".join(str(v) for v in row) + " |" for row in bench.itertuples(index=False)]
    f.write("\n".join(header + table) + "\n")
print(open("/kaggle/working/benchmark.md", encoding="utf-8").read())